# Local SME Underwriting Agents Notebook

Full Credit Memo workflow: Business Activity Analysis -> Credit Relationship -> Financial Analysis -> Credit Proposal -> Risk Assessment -> Credit Memo.

## 1. Setup

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import sys
import unicodedata
from dataclasses import asdict, dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any, Literal, TypedDict

from dotenv import load_dotenv
from IPython.display import Markdown, display
from langgraph.graph import END, StateGraph


def find_project_root(start: Path) -> Path:
    """Find the SME_ReAct_chatbot project root from the current cwd."""

    for candidate in [start, *start.parents]:
        extractor_path = candidate / "src" / "utils" / "extractors.py"
        if (candidate / "src").exists() and extractor_path.exists():
            return candidate
    return Path("/Users/uspro/Works/1. AI/1. RADAR/2. SME_creditmemo")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Canonical AI-agent package lives in notebooks/src (see docs/ARCHITECTURE.md).
NOTEBOOK_PACKAGE_DIR = PROJECT_ROOT / "src"
if str(NOTEBOOK_PACKAGE_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_PACKAGE_DIR))

load_dotenv(PROJECT_ROOT / ".env", override=True)

print(f"Project root: {PROJECT_ROOT}")

## 2. Runtime Parameters

Sửa `USER_PROMPT`, `TESTCASE_ID` và `INPUT_PATHS` trước khi chạy workflow.

In [ ]:
USER_PROMPT = "Hãy phân tích tài chính cho khách hàng này. Theo chương trình PLO"
TESTCASE_ID = "case_1"

# Có thể truyền file hoặc folder. Folder sẽ được scan đệ quy các file
# PDF/XLS/XLSX/CSV/TXT/MD.
INPUT_PATHS = [
    str(PROJECT_ROOT / "testing" / "samples" / TESTCASE_ID),
    # "/absolute/path/to/BCTC.pdf",
]

CONVERSATION_HISTORY = [
    # {"role": "user", "content": "..."},
    # {"role": "assistant", "content": "..."},
]

OUTPUT_DIR = PROJECT_ROOT / "logs"
MAX_FILES = 20
MAX_CHARS_PER_DOCUMENT = 120_000

## 3. LangSmith Tracing

In [ ]:
# LangSmith tracing — see underwriting.tracing (docs/ARCHITECTURE.md).
from src.tracing import run_supervisor_with_optional_trace


## 4. Shared Types

In [ ]:
# Shared types & helpers — see underwriting.types.
from src.types import (
    AgentName,
    WorkflowMode,
    DocumentAgentName,
    ClassifiedDocument,
    UnderwritingGraphState,
    to_dict_list,
    extract_text_from_agent_output,
    truncate_text,
)


## 5. LLM Config

In [ ]:
# LLM client factory & Config — extracted to underwriting.config.
from src.config import Config, build_llm

config = Config(
    decision_llm=build_llm("MODEL_DECISION", temperature=0.1),
    document_llm=build_llm("MODEL_DOCUMENT", temperature=0.5),
    conversation_llm=build_llm("MODEL_ECONOMY", temperature=0.8),
    analysis_llm=build_llm(
        "MODEL_ANALYZER",
        temperature=0.1,
        timeout_env="LLM_ANALYZE_TIMEOUT_SECONDS",
    ),
    credit_memo_llm=build_llm(
        "MODEL_CREDIT_MEMO",
        temperature=0.1,
        timeout_env="LLM_ANALYZE_TIMEOUT_SECONDS",
    ),
    hallucination_llm=build_llm(
        "MODEL_HALLUCINATION",
        temperature=0.0,
        timeout_env="LLM_ANALYZE_TIMEOUT_SECONDS",
    ),
    bctc_extraction_llm=build_llm(
        "MODEL_BCTC_EXTRACTION",
        temperature=0.0,
        timeout_env="LLM_ANALYZE_TIMEOUT_SECONDS",
    ),
    max_files=MAX_FILES,
    max_chars_per_document=MAX_CHARS_PER_DOCUMENT,
)

print(
    "LLM availability:",
    {
        "decision": bool(config.decision_llm),
        "document": bool(config.document_llm),
        "conversation": bool(config.conversation_llm),
        "analysis": bool(config.analysis_llm),
        "credit_memo": bool(config.credit_memo_llm),
        "hallucination": bool(config.hallucination_llm),
        "bctc_extraction": bool(config.bctc_extraction_llm),
    },
)

## 6. Tools

In [ ]:
# Database tools — see underwriting.tools.
from src.tools import (
    configure_database_executor,
    get_database_tools,
    DATABASE_TOOLS,
    FINANCIAL_DATABASE_TOOLS,
    BUSINESS_ACTIVITY_DATABASE_TOOLS,
    CREDIT_RELATIONSHIP_DATABASE_TOOLS,
    RISK_ASSESSMENT_DATABASE_TOOLS,
)


## 7. Document Extraction And Classification

In [ ]:
# Document extraction & classification — see underwriting.classification.
from src.agents.document_classification import (
    SUPPORTED_EXTENSIONS,
    VALID_DOCUMENT_AGENTS,
    compute_file_hash,
    resolve_input_path,
    discover_documents,
    normalize_text,
    document_type_scores,
    rule_classify_document,
)

# Document routing matrix (src/matrix/document_matrix.yaml): document type ->
# consuming agents. Loading it here surfaces a malformed matrix immediately
# instead of midway through a workflow run.
from src.agents.document_matrix import (
    agent_relevance_for_type,
    all_types,
    load_matrix,
    primary_agent_for_type,
)

_matrix = load_matrix()
print(
    f"Document matrix v{_matrix.version}: {len(all_types())} document types, "
    f"loan programs = {', '.join(_matrix.loan_programs)}"
)


## 8. Specialist Agents

In [ ]:
# Specialist agents & memo composer — see underwriting.agents.
from src.agents.specialist import (
    SpecialistAgent,
    BusinessActivityAnalysis,
    FinancialAnalysis,
    CreditRelationshipAnalysis,
    CreditProposalAnalysis,
    RiskAssessment,
    build_credit_memo,
    CreditMemoComposerAgent,
)


## 9. Optional Guardrails, Web Search, Hallucination Judge

In [ ]:
# Guardrails / web search / hallucination judge — see underwriting.guardrails.
from src.agents.guardrails import (
    LocalGuardrails,
    WebSearchProcessorAgent,
    HallucinationGuardrail,
)


## 10. Supervisor Agent

In [ ]:
# Supervisor orchestration — see underwriting.supervisor.
from src.agents.supervisor import Supervisor


## 11. Run Workflow

In [ ]:
from src.utils.common import show_graph
from src.utils.diagrams import mermaid_to_html

supervisor = Supervisor(config)

display(Markdown("### Workflow Graph"))
display(show_graph(supervisor.workflow_graph))

workflow_runner = globals().get("run_supervisor_with_optional_trace")
if workflow_runner:
    result = workflow_runner(
        supervisor,
        USER_PROMPT,
        INPUT_PATHS,
        CONVERSATION_HISTORY,
    )
else:
    result = supervisor.process(USER_PROMPT, INPUT_PATHS, CONVERSATION_HISTORY)

display(Markdown(mermaid_to_html(result["response"])))

print("\n--- Agent Name ---")
print(result["agent_name"])

print("\n--- Steps ---")
for step in result["steps"]:
    print("-", step)

# The loan program is read out of USER_PROMPT, so it must be visible: it selects which
# column of the document matrix decides each document's R/O relevance.
print("\n--- Loan Program ---")
_detection = result.get("loan_program_detection") or {}
if result.get("loan_program"):
    print(f"{result['loan_program']} "
          f"(matched \"{_detection.get('matched_alias')}\" "
          f"in the {_detection.get('source')})")
elif _detection.get("candidates"):
    print(f"AMBIGUOUS — the request names {', '.join(_detection['candidates'])}. "
          "Using the strongest relevance across all programs.")
    print("Name exactly one program in USER_PROMPT to pin it down.")
else:
    print("Not specified. Using the strongest relevance across all programs "
          "(documents can only be over-prioritised, never dropped).")
    print(f"Name one of {', '.join(_matrix.loan_programs)} in USER_PROMPT to pin it down.")

print("\n--- Hallucination Check ---")
check = result.get("hallucination_check") or {}
_claims = check.get("claims") or []
print("status:", check.get("status"), "| risk:", check.get("hallucination_risk"),
      "| action:", check.get("final_action"))
print("claims checked:", len(_claims),
      "| flagged:", len(check.get("unsupported_claims") or []),
      "| numeric errors:", len(check.get("numeric_errors") or []))
if check.get("summary"):
    print("summary:", check["summary"])

print("\n--- Document Classifications ---")
classification_keys = [
    "filename",
    "document_type",
    "document_group",
    "agent_relevance",
    "loan_program",
    "agent",
    "confidence",
    "reasoning",
    "extraction_status",
    "extraction_error",
    "classifier_error_type",
    "classifier_error",
]
for item in result["document_classifications"]:
    classification = {key: item.get(key) for key in classification_keys}
    print(json.dumps(classification, ensure_ascii=False, indent=2))

# Documents that matched no matrix row are shared with every agent, which is
# safe but imprecise — each one is a missing keyword or a missing document type.
_unmatched = [
    item["filename"]
    for item in result["document_classifications"]
    if not item.get("document_type")
]
if _unmatched:
    print(
        f"\nWARNING: {len(_unmatched)} document(s) matched no type in the "
        f"matrix and were shared with every agent: {', '.join(_unmatched)}"
    )


## 12. Save Artifacts

In [ ]:
run_id = "_".join([TESTCASE_ID, datetime.now().strftime("%Y%m%d_%H%M%S")])
run_dir = OUTPUT_DIR / run_id
run_dir.mkdir(parents=True, exist_ok=True)

(run_dir / "final_response.md").write_text(
    result["response"],
    encoding="utf-8",
)
(run_dir / "result.json").write_text(
    json.dumps(result, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(run_dir / "document_classifications.json").write_text(
    json.dumps(result["document_classifications"], ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(run_dir / "document_selections.json").write_text(
    json.dumps(result["document_selections"], ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(run_dir / "financial_metrics.json").write_text(
    json.dumps(result["financial_metrics"], ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(run_dir / "agent_outputs.json").write_text(
    json.dumps(result["sub_agent_outputs"], ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(run_dir / "hallucination_check.json").write_text(
    json.dumps(result["hallucination_check"], ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(run_dir / "financial_metrics.json").write_text(
    json.dumps(result.get("financial_metrics", {}), ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(run_dir / "bctc_extraction.json").write_text(
    json.dumps(
        {
            doc["filename"]: doc.get("bctc_extraction")
            for doc in result["document_classifications"]
            if doc.get("is_bctc")
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

# Export the final response (Markdown) to PDF via `markdown` + WeasyPrint.
# On macOS, WeasyPrint's Pango/Cairo bindings live under Homebrew and aren't on
# the default dynamic-linker path, so point DYLD_LIBRARY_PATH at them first.
import platform

if platform.system() == "Darwin":
    for brew_lib in ("/opt/homebrew/lib", "/usr/local/lib"):
        if os.path.isdir(brew_lib):
            os.environ["DYLD_LIBRARY_PATH"] = (
                brew_lib + ":" + os.environ.get("DYLD_LIBRARY_PATH", "")
            )

try:
    import markdown as md_lib
    from weasyprint import HTML

    from src.utils.diagrams import mermaid_to_html
    from src.utils.report_style import REPORT_CSS, tag_wide_tables

    # WeasyPrint has no JS, so mermaid blocks must become HTML before render.
    html_body = md_lib.markdown(
        mermaid_to_html(result["response"]),
        extensions=["tables", "fenced_code"],
    )
    # Wide financial tables get a smaller font so they fit the page box.
    html_body = tag_wide_tables(html_body)
    html_doc = (
        '<html><head><meta charset="utf-8"><style>'
        + REPORT_CSS
        + "</style></head><body>"
        + html_body
        + "</body></html>"
    )

    HTML(string=html_doc).write_pdf(str(run_dir / "final_response.pdf"))
    print(f"Saved PDF: {run_dir / 'final_response.pdf'}")
except ImportError:
    print(
        "Skipped PDF export — run `pip install markdown weasyprint` to enable it."
    )

print(f"Saved artifacts to: {run_dir}")